# Corr Model on Lattice ABPMultiScaleCNEEP2D applied to **Lattice Active Brownian Particles** (CTMC).GPU-accelerated simulation + training with `tqdm` progress bars.

In [ ]:
### for local server ###import sys, osCNEEP_V2_ROOT = os.path.abspath('/home/user1/CNEEP_v2')if CNEEP_V2_ROOT not in sys.path:    sys.path.append(CNEEP_V2_ROOT)

In [ ]:
sys.path.append(CNEEP_V2_ROOT)sys.path.append(os.path.join(CNEEP_V2_ROOT, 'data'))from argparse import Namespaceimport numpy as npimport torchfrom datetime import datetimefrom utils.sampler import CartesianSeqSamplerfrom tqdm import tqdmimport matplotlib.pyplot as pltfrom data.lattice_abp.core import LatticeABP

## 1. Hyperparameters

In [ ]:
## Hyper parameters#opt = Namespace()opt.device = "cuda" if torch.cuda.is_available() else "cpu"# alpha-NEEP paramsopt.alpha     = -0.5opt.beta      = 0.0opt.lam       = 0.0opt.threshold = 0.01opt.periodic    = Trueopt.positional  = Falseopt.latent_size = 10# trainingopt.n_iter           = 10000opt.train_batch_size = 2048opt.test_batch_size  = 2048opt.video_batch_size = 256opt.lr               = 1e-2opt.wd               = 1e-8opt.input_scalar     = 1opt.loss_scalar      = 1opt.scalar           = 1opt.clip_norm        = 1# MultiScale CNEEPopt.max_distance = 5opt.include_k0   = Falseopt.record_freq = 100opt.seed        = 3# dataset / modelopt.n_layer     = 2opt.n_channel   = 8opt.n_hidden    = 2opt.input_shape = (64, 64)opt.seq_len     = 2opt.val_ratio   = 0.2# Lattice ABP parametersabp_params = dict(    L=64,    v_plus=1.0,    v_zero=0.1,    v_minus=0.01,    D_rot=0.1,    density=0.5,    bc_mode='periodic',    device=opt.device,    seed=42,)# simulation parametersn_trajs     = 100    # ensemble size (M)n_steps     = 2000   # production steps per trajectoryburn_in     = 5000save_interval = 1    # save every steptau         = 0.01   # tau-leaping step size# test parametersn_trajs_test = 1n_steps_test = 10000torch.manual_seed(opt.seed)np.random.seed(opt.seed)# results folderresult_folder = os.path.join(CNEEP_V2_ROOT, 'results')current_result_folder = os.path.join(    result_folder, f"CorrLABP-{datetime.now().strftime('%Y-%m-%d-%H%M%S')}")os.makedirs(current_result_folder, exist_ok=True)current_checkpoint_path = os.path.join(current_result_folder, 'model_parameter.pth.tar')print(f"Device: {opt.device}")print(f"Results: {current_result_folder}")

## 2. Generate Lattice ABP Trajectories (Train)

In [ ]:
## Generate TRAIN trajectories using LatticeABP# Each call to simulate() produces (n_saved, B, L, L) occupancy snapshots.# We run multiple ensembles and stack them.#print(f"[INFO] Generating TRAIN trajectories (n_trajs={n_trajs}, n_steps={n_steps}, burn_in={burn_in})")sim_train = LatticeABP(**abp_params)result_train = sim_train.simulate(    B=n_trajs,    n_steps=n_steps,    burn_in=burn_in,    method='tau_leap',    tau=tau,    save_interval=save_interval,    show_progress=True,)# O_traj: (n_saved, B, L, L) -> we want (M, T, L, L)O_train = result_train['O_traj'].permute(1, 0, 2, 3).float()  # (M, T, L, L)print(f"[INFO] Train trajectories shape: {O_train.shape}")

## 3. Generate Trajectories (Test)

In [ ]:
## Generate TEST trajectories (longer, fewer ensembles)#print(f"[INFO] Generating TEST trajectories (n_trajs={n_trajs_test}, n_steps={n_steps_test})")sim_test = LatticeABP(**{**abp_params, 'seed': 123})result_test = sim_test.simulate(    B=n_trajs_test,    n_steps=n_steps_test,    burn_in=burn_in,    method='tau_leap',    tau=tau,    save_interval=save_interval,    show_progress=True,)O_test = result_test['O_traj'].permute(1, 0, 2, 3).float()print(f"[INFO] Test trajectories shape: {O_test.shape}")

## 4. Prepare Video Tensors

In [ ]:
## Prepare video tensors: (M, T, 1, Lx, Ly)#opt.M = O_train.shape[0]opt.L = O_train.shape[1]opt.M_test = O_test.shape[0]opt.L_test = O_test.shape[1]train_val_split_idx = int(opt.M * (1 - opt.val_ratio))M_train_new = train_val_split_idxM_val = opt.M - M_train_newtraj_train_new = O_train[:train_val_split_idx]traj_val = O_train[train_val_split_idx:]train_video = traj_train_new.to(opt.device).unsqueeze(2)   # (M_train, T, 1, Lx, Ly)val_video = traj_val.to(opt.device).unsqueeze(2)            # (M_val, T, 1, Lx, Ly)test_video = O_test.to(opt.device).unsqueeze(2)             # (M_test, T_test, 1, Lx, Ly)mean = torch.mean(train_video)std  = torch.std(train_video)transform = lambda x: (x - mean) * opt.input_scalar / stdprint(f"Train video tensor: {train_video.shape}")print(f"Val video tensor:   {val_video.shape}")print(f"Test video tensor:  {test_video.shape}")

## 5. Train MultiScaleCNEEP2D Model

In [ ]:
from models.NEEP_Corr_2D import MultiScaleCNEEP2Dfrom livelossplot import PlotLossesmodel = MultiScaleCNEEP2D(opt).to(opt.device)optim = torch.optim.Adam(model.parameters(), opt.lr, weight_decay=opt.wd)print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
## Training loop (MultiScaleCNEEP2D) — identical pattern to Corr_2D.ipynb#train_sampler = CartesianSeqSampler(    M_train_new, opt.L, opt.seq_len, opt.train_batch_size, device=opt.device)val_sampler = CartesianSeqSampler(    M_val, opt.L, opt.seq_len, opt.test_batch_size, device=opt.device, train=False)liveloss = PlotLosses()smoothing = 0.5smooth_train_loss = Nonesmooth_val_loss = Nonetrain_losses = []valid_losses = []best_val_loss = float('inf')best_checkpoint_path = os.path.join(current_result_folder, 'best_model_parameter.pth.tar')for it in tqdm(range(1, opt.n_iter + 1)):    # ── Train step ──    model.train()    batch = next(train_sampler)    b0 = batch[0].to(train_video.device)    slices = [train_video[(b0, batch[1][i].to(train_video.device))] for i in range(opt.seq_len)]    x = transform(torch.cat(slices, dim=1).float().to(opt.device))    J_all = model(x) / opt.scalar    ent_production = J_all.sum(dim=1)    optim.zero_grad()    if opt.alpha == 0:        loss = (- ent_production + (torch.exp(-ent_production) - 1)).mean()    else:        loss = (- (torch.exp(opt.alpha * ent_production) - 1) / opt.alpha            + (torch.exp(-(1 + opt.alpha) * ent_production) - 1) / (1 + opt.alpha)).mean()    (loss * opt.loss_scalar).backward()    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=opt.clip_norm)    optim.step()    train_losses.append(loss.item())    # ── Validation & checkpoint ──    if it % opt.record_freq == 0 or it == 1:        model.eval()        val_loss_acc = 0.0        n_val = 0        with torch.no_grad():            for vb in val_sampler:                vb0 = vb[0].to(val_video.device)                vslices = [val_video[(vb0, vb[1][i].to(val_video.device))] for i in range(opt.seq_len)]                vx = transform(torch.cat(vslices, dim=1).float().to(opt.device))                vJ = model(vx) / opt.scalar                v_ep = vJ.sum(dim=1)                if opt.alpha == 0:                    vloss = (- v_ep + (torch.exp(-v_ep) - 1)).sum().item()                else:                    vloss = (- (torch.exp(opt.alpha * v_ep) - 1) / opt.alpha                        + (torch.exp(-(1 + opt.alpha) * v_ep) - 1) / (1 + opt.alpha)).sum().item()                val_loss_acc += vloss                n_val += vx.shape[0]        avg_val = val_loss_acc / n_val        valid_losses.append(avg_val)        state = {            'settings': opt.__dict__,            'state_dict': model.state_dict(),            'optimizer': optim.state_dict(),            'iteration': it,        }        torch.save(state, current_checkpoint_path)        if avg_val < best_val_loss:            best_val_loss = avg_val            torch.save(state, best_checkpoint_path)        if smooth_train_loss is None:            smooth_train_loss = loss.item()            smooth_val_loss = avg_val        else:            smooth_train_loss = smoothing * smooth_train_loss + (1 - smoothing) * loss.item()            smooth_val_loss = smoothing * smooth_val_loss + (1 - smoothing) * avg_val        liveloss.update({'train_loss': smooth_train_loss, 'val_loss': smooth_val_loss})        liveloss.send()print('Training finished.')print(f'Checkpoint: {current_checkpoint_path}')

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))axes[0].plot(train_losses[100:]); axes[0].set_title('Train Loss')axes[1].plot(valid_losses); axes[1].set_title('Valid Loss')for ax in axes: ax.set_xlabel('Iteration')plt.tight_layout()plt.savefig(f'{current_result_folder}/training_curves.png', dpi=150)plt.show()

## 6-b. Model Selection

In [ ]:
load_best = Truebest_checkpoint_path = os.path.join(current_result_folder, 'best_model_parameter.pth.tar')if load_best and os.path.exists(best_checkpoint_path):    print("Loading BEST model (lowest validation loss)...")    checkpoint = torch.load(best_checkpoint_path, map_location=opt.device)else:    print("Loading FINAL iteration model...")    checkpoint = torch.load(current_checkpoint_path, map_location=opt.device)model.load_state_dict(checkpoint['state_dict'])model.eval()print(f"Loaded model from iteration {checkpoint.get('iteration', 'Unknown')}")

## 7. Ground Truth EPR vs Predicted EPR

In [ ]:
## Ground truth EPR from Lattice ABP transition rates#stride = opt.seq_len - 1n_windows = (opt.L_test - 1) // strideprint(f"[INFO] Computing GT EPR on TEST data (n_windows={n_windows}) ...")gt_total_epr = np.zeros(n_windows)Lx = abp_params['L']gt_epr_map_sum = np.zeros((Lx, Lx))# Reconstruct O and E trajectories for GT EPR computationO_test_traj = result_test['O_traj']  # (n_saved, B, L, L)E_test_traj = result_test['E_traj']for w in tqdm(range(n_windows)):    t_start = w * stride    window_epr_map = np.zeros((Lx, Lx))    for s in range(stride):        t = t_start + s        O_t = O_test_traj[t].to(opt.device)        E_t = E_test_traj[t].to(opt.device)        epr_map = sim_test.compute_local_epr(O_t, E_t)        mean_epr_map = epr_map.mean(dim=0)        window_epr_map += mean_epr_map.cpu().numpy()    gt_epr_map_sum += window_epr_map    gt_total_epr[w] = np.sum(window_epr_map)gt_epr_maps = (gt_epr_map_sum / n_windows)[np.newaxis, ...]print(f"GT mean EPR rate: {(gt_total_epr / stride).mean():.6e}")print(f"GT windows: {n_windows}")

In [ ]:
## Compare total Predicted EP with GT EPR#model.eval()pred_total_ep = []test_sampler = CartesianSeqSampler(    opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size,    device=opt.device, train=False)with torch.no_grad():    for batch in test_sampler:        b0 = batch[0].to(test_video.device)        slices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]        x = transform(torch.cat(slices, dim=1).float().to(opt.device))        J_all = model(x) / opt.scalar        total_ep = J_all.sum(dim=1) * (abp_params['L']**2)        pred_total_ep.append(total_ep.cpu().numpy())pred_total_ep = np.concatenate(pred_total_ep)min_len = min(len(gt_total_epr), len(pred_total_ep))time_axis = np.arange(min_len) * taufig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)axes[0].plot(time_axis, gt_total_epr[:min_len], lw=0.5, alpha=0.6, label='GT EPR')axes[0].plot(time_axis, pred_total_ep[:min_len] / tau, lw=0.5, alpha=0.6, label='Pred EP/tau')axes[0].set_ylabel('EPR'); axes[0].legend(); axes[0].set_title('Instantaneous EPR')axes[1].plot(time_axis, np.cumsum(gt_total_epr[:min_len] * tau), label='GT Cumul EP')axes[1].plot(time_axis, np.cumsum(pred_total_ep[:min_len]), label='Pred Cumul EP')axes[1].set_ylabel('Cumulative EP'); axes[1].legend()window = 100gt_smooth = np.convolve(gt_total_epr[:min_len], np.ones(window)/window, mode='same')pred_smooth = np.convolve(pred_total_ep[:min_len]/tau, np.ones(window)/window, mode='same')axes[2].plot(time_axis, gt_smooth, label='GT (Smooth)')axes[2].plot(time_axis, pred_smooth, label='Pred (Smooth)')axes[2].set_ylabel('EPR (Running Avg)'); axes[2].set_xlabel('Time'); axes[2].legend()plt.tight_layout()plt.savefig(f'{current_result_folder}/epr_timeseries.png', dpi=150)plt.show()print(f'GT mean EPR:   {gt_total_epr[:min_len].mean():.6e}')print(f'Pred mean EPR: {(pred_total_ep[:min_len]/tau).mean():.6e}')

## 8. Spatial EP Map

In [ ]:
## Visualize Local EP Map 2D Heatmaps#model.eval()test_sampler_one = CartesianSeqSampler(    opt.M_test, opt.L_test, opt.seq_len, 1,    device=opt.device, train=False)ens_idx, traj_idx = next(test_sampler_one)b0 = ens_idx.to(test_video.device)slices = [test_video[(b0, traj_idx[i].to(test_video.device))] for i in range(opt.seq_len)]x = transform(torch.cat(slices, dim=1).float().to(opt.device))with torch.no_grad():    maps = model(x, return_maps=True) / opt.scalarsample_idx = 0Lx = abp_params['L']phi_t = x[sample_idx, 0].cpu().numpy()pred_map_k = maps[sample_idx].cpu().numpy() / taupred_total_map = pred_map_k.sum(axis=0)gt_map = gt_epr_maps[0]fig, axes = plt.subplots(1, 3, figsize=(15, 4))im0 = axes[0].imshow(phi_t.T, origin='lower', cmap='viridis')axes[0].set_title('Input State (Occupancy)'); fig.colorbar(im0, ax=axes[0])vmax = max(np.abs(gt_map).max(), np.abs(pred_total_map).max(), 1e-12)im1 = axes[1].imshow(gt_map.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)axes[1].set_title('GT EPR Map'); fig.colorbar(im1, ax=axes[1])im2 = axes[2].imshow(pred_total_map.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)axes[2].set_title('Predicted Total EP Map'); fig.colorbar(im2, ax=axes[2])plt.tight_layout()plt.savefig(f'{current_result_folder}/local_ep_map_2d.png', dpi=150)plt.show()

## 9. EP Spectrum

In [ ]:
## EP spectrum: mean J_k for each distance k#model.eval()all_J = []test_sampler = CartesianSeqSampler(    opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size,    device=opt.device, train=False)with torch.no_grad():    for batch in test_sampler:        b0 = batch[0].to(test_video.device)        slices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]        x = transform(torch.cat(slices, dim=1).float().to(opt.device))        J = model(x) / opt.scalar        all_J.append(J.cpu().numpy() * (abp_params['L']**2))all_J = np.concatenate(all_J, axis=0)mean_J = all_J.mean(axis=0)std_J  = all_J.std(axis=0)distances_list = list(range(0 if getattr(opt, "include_k0", True) else 1, opt.max_distance + 1))distances = np.array(distances_list)fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].bar(distances, mean_J / tau, yerr=std_J / tau / np.sqrt(len(all_J)),            capsize=3, alpha=0.7, color='steelblue')axes[0].set_xlabel('Kernel $k$')axes[0].set_ylabel(r'$\langle \dot{S}_k \rangle$')axes[0].set_title('EP Spectrum: EP rate by correlation distance')axes[0].set_xticks(distances)cum_J = np.cumsum(mean_J)axes[1].plot(distances, cum_J / tau, 'o-', color='darkorange')axes[1].set_xlabel('Kernel $k$')axes[1].set_ylabel(r'$\sum_{k} \langle \dot{S}_k \rangle$')axes[1].set_title('Cumulative EP rate')axes[1].set_xticks(distances)plt.tight_layout()plt.savefig(f'{current_result_folder}/ep_spectrum.png', dpi=150)plt.show()print(f'Total estimated EP rate: {mean_J.sum() / tau:.6e}')for idx, k in enumerate(distances_list):    print(f'  k={k}: J_k / tau = {mean_J[idx] / tau:.6e}  '          f'({100 * mean_J[idx] / mean_J.sum():.1f}%)')